# grads-dict-accumulate-parents — worked example 3: grads dict: full two-node reverse pass with dict accumulation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grads-dict-accumulate-parents`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The reverse pass walks nodes in topological reverse order. At each node, it reads the outgoing gradient from `grads[node]` (set by the previous iteration), dispatches the per-argument backward function to produce per-parent contributions, and calls the accumulation function to add those contributions to the parent entries in `grads`. The loop terminates when all nodes have been processed; any leaves in `grads` at that point hold the final `dL/dleaf` values.

## Worked solution

**Step 1 — set up a simple chain: loss = relu(w * x).** Two internal nodes: `z = w * x` and `loss = relu(z)`. We'll run the reverse pass manually.

**Step 2 — seed with loss gradient.** `grads[loss_node] = tensor(1.0)` — the upstream gradient at the root is always 1 for a scalar loss.

**Step 3 — propagate through relu.** The relu backward says `d_loss/dz = d_loss/d_loss * (z > 0)`. Accumulate into `grads[z_node]`.

**Step 4 — propagate through multiply.** The mul backward gives two contributions: `d_loss/dw = d_loss/dz * x` and `d_loss/dx = d_loss/dz * w`. Accumulate both.

**Step 5 — verify against autograd.** Run the same computation with `requires_grad=True` and confirm the grads dict values match.

In [ ]:
import torch as t

t.manual_seed(5)

def accumulate_into_grads(grads, contributions):
    for parent, g in contributions:
        grads[parent] = grads.get(parent, 0) + g

class Node:
    def __init__(self, name, val):
        self.name = name
        self.val = val

# Forward pass values
w_v = t.tensor(2.0)
x_v = t.tensor(3.0)
z_v = w_v * x_v             # z = 6.0
loss_v = t.relu(z_v)        # loss = 6.0 (z > 0, so relu is identity here)

# Create node objects
w_node   = Node('w', w_v)
x_node   = Node('x', x_v)
z_node   = Node('z', z_v)
loss_node = Node('loss', loss_v)

grads = {}
# Seed: dL/dloss = 1
grads[loss_node] = t.tensor(1.0)

# Step 1: relu backward for z_node (parent of loss_node)
# d(relu)/dz = 1 if z > 0 else 0
relu_back_z = grads[loss_node] * (z_v > 0).float()
accumulate_into_grads(grads, [(z_node, relu_back_z)])

# Step 2: mul backward for z = w * x
# dz/dw = x, dz/dx = w
mul_back_w = grads[z_node] * x_v
mul_back_x = grads[z_node] * w_v
accumulate_into_grads(grads, [(w_node, mul_back_w), (x_node, mul_back_x)])

print(f"grads[w]: {grads[w_node].item()}")  # d(relu(w*x))/dw = x = 3.0
print(f"grads[x]: {grads[x_node].item()}")  # d(relu(w*x))/dx = w = 2.0

# Verify with real autograd
w_r = t.tensor(2.0, requires_grad=True)
x_r = t.tensor(3.0, requires_grad=True)
loss_r = t.relu(w_r * x_r)
loss_r.backward()
assert abs(grads[w_node].item() - w_r.grad.item()) < 1e-5
assert abs(grads[x_node].item() - x_r.grad.item()) < 1e-5
print("Matches PyTorch autograd!")